# Caso 1: Audio a Texto con Whisper (Parte 1)

En este notebook, exploraremos cómo cargar y preparar datos de audio para su posterior transcripción, optimizando el uso de memoria para Colab.

## 1. Configuración del Entorno

Primero, instalamos las bibliotecas necesarias para trabajar con datos de audio y modelos de transcripción.

In [ ]:
!pip install datasets openai-whisper matplotlib numpy pandas seaborn librosa pydub tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 9.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
... [output truncated for repo] ...


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from datasets import load_dataset, Audio
from IPython.display import display, Audio as IPythonAudio
from tqdm.notebook import tqdm

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

## 2. Creación de Directorios

Vamos a crear los directorios necesarios para almacenar los archivos de audio y los resultados.

In [ ]:
# Crear directorios para archivos de audio y resultados
audio_dir = "audio_files"
results_dir = "results"

os.makedirs(audio_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print(f"Directorios creados: {audio_dir}, {results_dir}")

Directorios creados: audio_files, results


## 3. Carga de Datos de Audio (Optimizada para Colab)

Vamos a cargar datos de audio de manera eficiente para evitar problemas de memoria en Colab. Ofrecemos dos opciones:

### Opción 1: Usar un subconjunto pequeño del dataset Common Voice

In [ ]:
def load_small_subset():
    try:
        print("Cargando un pequeño subconjunto del dataset Common Voice...")
        # Cargar solo los primeros 50 ejemplos
        dataset = load_dataset(
            "mozilla-foundation/common_voice_11_0",
            "es",
            split="test[:50]",  # Solo los primeros 50 ejemplos
            streaming=True       # Modo streaming para no cargar todo en memoria
        )

        # Convertir a una lista pequeña para trabajar con ella
        dataset_samples = []
        for i, example in enumerate(dataset):
            if i >= 10:  # Solo tomar 10 ejemplos
                break
            dataset_samples.append(example)

        print(f"Cargados {len(dataset_samples)} ejemplos del dataset Common Voice")
        return dataset_samples
    except Exception as e:
        print(f"Error al cargar el subconjunto: {e}")
        return None

### Opción 2: Usar un dataset más pequeño y común

In [ ]:
def load_smaller_dataset():
    try:
        print("Intentando cargar un dataset más pequeño...")
        # Intentar cargar un dataset más pequeño y común
        dataset = load_dataset("PolyAI/minds14", "es-ES", split="train[:20]")
        print(f"Dataset más pequeño cargado con {len(dataset)} ejemplos")
        return dataset
    except Exception as e:
        print(f"Error al cargar el dataset más pequeño: {e}")
        return None

### Opción 3: Crear ejemplos simulados

In [ ]:
def create_simulated_examples():
    print("Creando ejemplos simulados...")

    # Ejemplos de texto en español
    ejemplos_texto = [
        "Bienvenidos al tutorial sobre conversión de audio a texto utilizando Python.",
        "La inteligencia artificial está transformando la forma en que interactuamos con la tecnología.",
        "Los sistemas de reconocimiento de voz permiten crear experiencias más accesibles e inclusivas.",
        "La calidad de las transcripciones ha mejorado significativamente en los últimos años gracias a los avances en aprendizaje profundo.",
        "Combinar reconocimiento de voz y procesamiento de lenguaje natural permite crear aplicaciones más inteligentes."
    ]

    # Crear ejemplos simulados
    simulated_examples = []
    for i, texto in enumerate(ejemplos_texto):
        # Crear un array de audio simulado (ruido blanco)
        sr = 16000  # Frecuencia de muestreo típica
        duration = random.uniform(2.0, 5.0)  # Duración entre 2 y 5 segundos
        n_samples = int(sr * duration)
        audio_array = np.random.randn(n_samples).astype(np.float32) * 0.1  # Ruido blanco de baja amplitud

        simulated_examples.append({
            'id': str(i),
            'audio': {
                'array': audio_array,
                'sampling_rate': sr
            },
            'sentence': texto,
            'path': f"simulated_audio_{i}.wav"
        })

    print(f"Creados {len(simulated_examples)} ejemplos simulados")
    return simulated_examples

### Intentar cargar datos con las diferentes opciones

In [ ]:
# Intentar las diferentes opciones en orden
dataset_samples = None

# Primero intentar con un subconjunto pequeño del dataset original
try:
    dataset_samples = load_small_subset()
except Exception as e:
    print(f"Error al cargar subconjunto pequeño: {e}")

# Si falla, intentar con un dataset más pequeño
if dataset_samples is None or len(dataset_samples) == 0:
    try:
        dataset_samples = load_smaller_dataset()
    except Exception as e:
        print(f"Error al cargar dataset alternativo: {e}")

# Si todo falla, crear ejemplos simulados
if dataset_samples is None or len(dataset_samples) == 0:
    dataset_samples = create_simulated_examples()

print(f"\nTotal de ejemplos disponibles: {len(dataset_samples)}")

Cargando un pequeño subconjunto del dataset Common Voice...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/14.4k [00:00<?, ?B/s]

common_voice_11_0.py:   0%|          | 0.00/8.13k [00:00<?, ?B/s]

languages.py:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

...

minds14.py:   0%|          | 0.00/5.83k [00:00<?, ?B/s]

The repository for PolyAI/minds14 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/PolyAI/minds14.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


MInDS-14.zip:   0%|          | 0.00/471M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset más pequeño cargado con 20 ejemplos

Total de ejemplos disponibles: 20


## 4. Selección de Ejemplos Aleatorios

Vamos a seleccionar 5 ejemplos aleatorios del conjunto de datos.

In [ ]:
import random

# Seleccionar 5 ejemplos aleatorios
def select_random_examples(samples, num_examples=5, seed=42):
    # Establecer semilla para reproducibilidad
    random.seed(seed)

    # Convertir a lista si no lo es
    sample_list = list(samples)

    # Seleccionar ejemplos aleatorios
    if len(sample_list) <= num_examples:
        selected = sample_list
    else:
        selected = random.sample(sample_list, num_examples)

    print(f"Seleccionados {len(selected)} ejemplos aleatorios")
    return selected

# Seleccionar ejemplos
selected_examples = select_random_examples(dataset_samples)

Seleccionados 5 ejemplos aleatorios


## 5. Exploración de los Ejemplos Seleccionados

Vamos a explorar los ejemplos seleccionados para entender mejor los datos.

In [ ]:
# Mostrar información de los ejemplos seleccionados
for i, example in enumerate(selected_examples):
    print(f"\nEjemplo {i+1}:")

    # Verificar si el ejemplo tiene el campo 'sentence' o 'text'
    if 'sentence' in example:
        print(f"  Texto: {example['sentence']}")
    elif 'text' in example:
        print(f"  Texto: {example['text']}")
    else:
        print("  Texto: No disponible")

    # Verificar si tenemos datos de audio reales o simulados
    if 'audio' in example and isinstance(example['audio'], dict) and 'array' in example['audio']:
        audio_array = example['audio']['array']
        sampling_rate = example['audio']['sampling_rate']
        duration = len(audio_array) / sampling_rate
        print(f"  Duración del audio: {duration:.2f} segundos")
        print(f"  Frecuencia de muestreo: {sampling_rate} Hz")
    else:
        print("  Datos de audio no disponibles en formato estándar")


Ejemplo 1:
  Texto: No disponible
  Duración del audio: 11.43 segundos
  Frecuencia de muestreo: 8000 Hz

Ejemplo 2:
  Texto: No disponible
  Duración del audio: 17.07 segundos
  Frecuencia de muestreo: 8000 Hz

Ejemplo 3:
  Texto: No disponible
  Duración del audio: 12.54 segundos
  Frecuencia de muestreo: 8000 Hz

Ejemplo 4:
  Texto: No disponible
  Duración del audio: 10.24 segundos
  Frecuencia de muestreo: 8000 Hz

Ejemplo 5:
  Texto: No disponible
  Duración del audio: 12.03 segundos
  Frecuencia de muestreo: 8000 Hz


## 6. Guardar Archivos de Audio

Vamos a guardar los archivos de audio seleccionados para su posterior procesamiento.

In [ ]:
# Función para guardar un archivo de audio
def save_audio_file(audio_array, sampling_rate, output_file):
    try:
        # Intentar con soundfile
        try:
            import soundfile as sf
            sf.write(output_file, audio_array, sampling_rate)
            return True
        except Exception as e1:
            print(f"Error con soundfile: {e1}")

            # Intentar con scipy
            try:
                from scipy.io import wavfile
                wavfile.write(output_file, sampling_rate, audio_array)
                return True
            except Exception as e2:
                print(f"Error con scipy: {e2}")

                # Como último recurso, guardar como numpy array
                try:
                    np.save(output_file.replace(".wav", ".npy"), audio_array)
                    print(f"Guardado como array numpy: {output_file.replace('.wav', '.npy')}")
                    return True
                except Exception as e3:
                    print(f"Error al guardar como numpy: {e3}")
                    return False
    except Exception as e:
        print(f"Error general al guardar audio: {e}")
        return False

In [ ]:
# Guardar los archivos de audio seleccionados
audio_files = []

for i, example in enumerate(selected_examples):
    output_file = os.path.join(audio_dir, f"audio_{i+1}.wav")

    # Obtener el texto del ejemplo
    if 'sentence' in example:
        text = example['sentence']
    elif 'text' in example:
        text = example['text']
    else:
        text = f"Texto del ejemplo {i+1} (no disponible)"

    # Verificar si tenemos datos de audio reales o simulados
    if 'audio' in example and isinstance(example['audio'], dict) and 'array' in example['audio']:
        # Guardar el archivo de audio
        success = save_audio_file(
            example['audio']['array'],
            example['audio']['sampling_rate'],
            output_file
        )

        if success:
            audio_files.append({
                'id': i,
                'file_path': output_file,
                'original_text': text,
                'duration': len(example['audio']['array']) / example['audio']['sampling_rate']
            })
            print(f"Archivo de audio guardado: {output_file}")
    else:
        print(f"No se pudo guardar el audio para el ejemplo {i+1} (datos de audio no disponibles)")
        # Crear un registro de todos modos para mantener la consistencia
        audio_files.append({
            'id': i,
            'file_path': output_file,
            'original_text': text,
            'duration': 0.0  # Duración desconocida
        })

Archivo de audio guardado: audio_files/audio_1.wav
Archivo de audio guardado: audio_files/audio_2.wav
Archivo de audio guardado: audio_files/audio_3.wav
Archivo de audio guardado: audio_files/audio_4.wav
Archivo de audio guardado: audio_files/audio_5.wav


## 7. Crear DataFrame con Información de Audio

Vamos a crear un DataFrame con la información de los archivos de audio para facilitar su procesamiento.

In [ ]:
# Crear DataFrame con información de audio
df_audio = pd.DataFrame(audio_files)
df_audio

## 8. Guardar DataFrame para Uso Posterior

Vamos a guardar el DataFrame para su uso en los siguientes notebooks.

In [ ]:
# Guardar DataFrame a CSV
csv_file = os.path.join(results_dir, "audio_files.csv")
df_audio.to_csv(csv_file, index=False)
print(f"DataFrame guardado en: {csv_file}")

DataFrame guardado en: results/audio_files.csv


## 9. Visualización de Archivos de Audio (Opcional)

Si los archivos de audio se guardaron correctamente, podemos visualizarlos y reproducirlos.

In [ ]:
# Función para visualizar forma de onda
def plot_audio_waveform(file_path, title):
    try:
        if not os.path.exists(file_path):
            # Verificar si existe una versión .npy del archivo
            npy_file = file_path.replace(".wav", ".npy")
            if os.path.exists(npy_file):
                # Cargar el array numpy
                y = np.load(npy_file)
                sr = 16000  # Frecuencia de muestreo típica
            else:
                print(f"Ni el archivo {file_path} ni {npy_file} existen.")
                return None
        else:
            # Cargar el archivo de audio
            y, sr = librosa.load(file_path)

        plt.figure(figsize=(12, 4))
        librosa.display.waveshow(y, sr=sr)
        plt.title(f'Forma de onda - {title}')
        plt.xlabel('Tiempo (s)')
        plt.ylabel('Amplitud')
        plt.tight_layout()
        plt.show()

        return IPythonAudio(data=y, rate=sr)
    except Exception as e:
        print(f"Error al visualizar audio: {e}")
        return None

In [ ]:
# Visualizar y reproducir cada archivo de audio (si existen)
for _, row in df_audio.iterrows():
    file_path = row['file_path']
    npy_file = file_path.replace(".wav", ".npy")

    if os.path.exists(file_path) or os.path.exists(npy_file):
        print(f"\nAudio {row['id']}: {row['original_text']}")
        print(f"Duración: {row['duration']:.2f} segundos")

        audio_player = plot_audio_waveform(file_path, f"Ejemplo {row['id']}")
        if audio_player:
            display(audio_player)
    else:
        print(f"\nAudio {row['id']}: Archivo no disponible")


Audio 0: Texto del ejemplo 1 (no disponible)
Duración: 11.43 segundos



Audio 1: Texto del ejemplo 2 (no disponible)
Duración: 17.07 segundos



Audio 2: Texto del ejemplo 3 (no disponible)
Duración: 12.54 segundos



Audio 3: Texto del ejemplo 4 (no disponible)
Duración: 10.24 segundos



Audio 4: Texto del ejemplo 5 (no disponible)
Duración: 12.03 segundos


## 10. Resumen

En este notebook, hemos:

1. Configurado el entorno para trabajar con datos de audio
2. Implementado estrategias diferentes para cargar datos de audio de manera eficiente en Colab:
   - Cargar un pequeño subconjunto del dataset Common Voice
   - Usar un dataset alternativo más pequeño (PolyAI/minds14)
   - Crear ejemplos simulados si las opciones anteriores fallan
3. Seleccionado 5 ejemplos aleatorios
4. Guardado los archivos de audio para su posterior procesamiento
5. Creado un DataFrame con la información de los archivos de audio
6. Visualizado y reproducido los archivos de audio (si están disponibles)

En el siguiente notebook, utilizaremos estos archivos de audio para realizar la transcripción con Whisper.

# Caso 1: Audio to Text con Whisper y Datasets de Hugging Face (Parte 2)

En esta segunda parte, utilizaremos la API de Whisper de OpenAI para transcribir los 5 audios aleatorios que seleccionamos en la Parte 1.

## 1. Configuración para Transcripción con Whisper

Primero, configuramos las funciones necesarias para utilizar la API de Whisper.

In [ ]:
import os
import openai
import pandas as pd
import time
from tqdm.notebook import tqdm

# Si estás ejecutando este notebook por separado, descomenta y ejecuta estas líneas
# import getpass
# api_key = getpass.getpass("Introduce tu clave API de OpenAI: ")
# openai.api_key = api_key

# Función para transcribir audio con Whisper
def transcribe_audio(file_path, model="whisper-1", language="es", max_retries=3, retry_delay=5):
    for attempt in range(max_retries):
        try:
            with open(file_path, "rb") as audio_file:
                response = openai.Audio.transcribe(
                    model=model,
                    file=audio_file,
                    language=language
                )
            return response.text
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"Error: {e}. Reintentando en {retry_delay} segundos...")
                time.sleep(retry_delay)
            else:
                print(f"Error después de {max_retries} intentos: {e}")
                return f"Error: {e}"

# Función alternativa que utiliza requests directamente (por si la anterior falla)
def transcribe_audio_requests(file_path, api_key, model="whisper-1", language="es"):
    import requests

    url = "https://api.openai.com/v1/audio/transcriptions"
    headers = {
        "Authorization": f"Bearer {api_key}"
    }

    with open(file_path, "rb") as audio_file:
        files = {
            "file": (os.path.basename(file_path), audio_file, "audio/wav"),
            "model": (None, model),
            "language": (None, language)
        }
        response = requests.post(url, headers=headers, files=files)

    if response.status_code == 200:
        return response.json()["text"]
    else:
        return f"Error: {response.status_code} - {response.text}"

## 2. Transcripción de los Audios Seleccionados

Ahora, transcribiremos los 5 audios aleatorios que seleccionamos y guardamos en la Parte 1.

In [ ]:
# Verificar que tenemos los archivos de audio
temp_dir = "audio_files"
if not os.path.exists(temp_dir):
    print("El directorio de archivos temporales no existe. Ejecuta primero la Parte 1.")
else:
    audio_files = [os.path.join(temp_dir, f) for f in os.listdir(temp_dir) if f.endswith(".wav")]
    print(f"Encontrados {len(audio_files)} archivos de audio:")
    for file in audio_files:
        print(f"  - {file}")

Encontrados 5 archivos de audio:
  - audio_files/audio_5.wav
  - audio_files/audio_3.wav
  - audio_files/audio_4.wav
  - audio_files/audio_1.wav
  - audio_files/audio_2.wav


In [ ]:
# Si estás ejecutando este notebook por separado, crea un DataFrame con la información de los archivos
# Descomenta y ejecuta estas líneas si es necesario

audio_files_df = pd.DataFrame([
    {'index': i, 'path': file, 'text': 'Texto original no disponible'}
    for i, file in enumerate(audio_files)
])

In [ ]:
import openai
from openai import OpenAI
from pathlib import Path

# Crea una instancia del cliente
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def transcribe_audio(file_path):
    with open(file_path, "rb") as f:
        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            file=f
        )
    return transcription.text

In [ ]:
# Transcribir los audios con Whisper
transcriptions = []

try:
    # Intentar cargar audio_files_df de la Parte 1
    audio_files_df
except NameError:
    # Si no está definido, crear uno nuevo
    audio_files = [os.path.join(temp_dir, f) for f in os.listdir(temp_dir) if f.endswith(".wav")]
    audio_files_df = pd.DataFrame([
        {'index': i, 'path': file, 'text': 'Texto original no disponible'}
        for i, file in enumerate(audio_files)
    ])

for _, row in tqdm(audio_files_df.iterrows(), total=len(audio_files_df), desc="Transcribiendo audios"):
    file_path = row['path']
    original_text = row['text']

    # Transcribir con Whisper
    try:
        transcribed_text = transcribe_audio(file_path)
    except Exception as e:
        print(f"Error con la API de OpenAI: {e}. Intentando método alternativo...")
        try:
            transcribed_text = transcribe_audio_requests(file_path, openai.api_key)
        except Exception as e2:
            print(f"Error con el método alternativo: {e2}")
            transcribed_text = "Error en la transcripción"

    transcriptions.append({
        'index': row['index'],
        'file_path': file_path,
        'original_text': original_text,
        'transcribed_text': transcribed_text
    })

# Crear DataFrame con los resultados
transcriptions_df = pd.DataFrame(transcriptions)
transcriptions_df

Transcribiendo audios:   0%|          | 0/5 [00:00<?, ?it/s]

## 3. Comparación de Resultados

Vamos a comparar el texto original con la transcripción generada por Whisper.

In [ ]:
# Función para calcular la similitud entre dos textos
def calculate_similarity(text1, text2):
    # Normalizar textos
    text1 = text1.lower().strip()
    text2 = text2.lower().strip()

    # Método simple: proporción de caracteres coincidentes
    from difflib import SequenceMatcher
    return SequenceMatcher(None, text1, text2).ratio()

# Calcular similitud para cada par de textos
similarities = []
for _, row in transcriptions_df.iterrows():
    if row['original_text'] != 'Texto original no disponible':
        similarity = calculate_similarity(row['original_text'], row['transcribed_text'])
        similarities.append({
            'index': row['index'],
            'original_text': row['original_text'],
            'transcribed_text': row['transcribed_text'],
            'similarity': similarity
        })

# Crear DataFrame con los resultados
if similarities:
    similarities_df = pd.DataFrame(similarities)
    similarities_df['similarity_percentage'] = similarities_df['similarity'] * 100
    similarities_df = similarities_df.sort_values('similarity', ascending=False)
    similarities_df

In [ ]:
# Visualizar la similitud entre textos originales y transcripciones
if 'similarities_df' in locals() and not similarities_df.empty:
    plt.figure(figsize=(10, 6))
    bars = plt.bar(similarities_df['index'], similarities_df['similarity_percentage'])

    # Colorear las barras según el nivel de similitud
    for i, bar in enumerate(bars):
        similarity = similarities_df.iloc[i]['similarity']
        if similarity >= 0.8:
            bar.set_color('green')
        elif similarity >= 0.6:
            bar.set_color('orange')
        else:
            bar.set_color('red')

    plt.axhline(y=80, color='green', linestyle='--', alpha=0.7, label='Excelente (≥80%)')
    plt.axhline(y=60, color='orange', linestyle='--', alpha=0.7, label='Aceptable (≥60%)')

    plt.xlabel('Índice del Audio')
    plt.ylabel('Similitud (%)')
    plt.title('Similitud entre Texto Original y Transcripción')
    plt.ylim(0, 105)
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Estadísticas de similitud
    print(f"Similitud promedio: {similarities_df['similarity'].mean() * 100:.2f}%")
    print(f"Similitud máxima: {similarities_df['similarity'].max() * 100:.2f}%")
    print(f"Similitud mínima: {similarities_df['similarity'].min() * 100:.2f}%")

## 4. Análisis Detallado de las Transcripciones

Vamos a examinar en detalle cada transcripción para identificar patrones y errores comunes.

In [ ]:
# Mostrar cada par de textos (original y transcripción) para análisis detallado
for i, row in transcriptions_df.iterrows():
    print(f"\n--- Audio {row['index']} ---")
    print(f"Original: {row['original_text']}")
    print(f"Whisper:  {row['transcribed_text']}")

    if row['original_text'] != 'Texto original no disponible':
        similarity = calculate_similarity(row['original_text'], row['transcribed_text'])
        print(f"Similitud: {similarity * 100:.2f}%")

        # Identificar diferencias
        import difflib
        d = difflib.Differ()
        diff = list(d.compare(row['original_text'].lower().split(), row['transcribed_text'].lower().split()))
        print("\nDiferencias:")
        print(' '.join(diff))


--- Audio 0 ---
Original: Texto original no disponible
Whisper:  Hola, buenas. Quiero utilizar la aplicación del banco, pero no consigo que funcione en condiciones. No sé si es que está en mantenimiento o tengo algún problema yo. ¿Me podríais ayudar?

--- Audio 1 ---
Original: Texto original no disponible
Whisper:  Hola, estoy intentando acceder a la aplicación de mi banco pero no carga correctamente. Me aparece un mensaje como que está siendo reparada y no puedo acceder.

--- Audio 2 ---
Original: Texto original no disponible
Whisper:  Hola, buenos días. Quería informaros porque no me funciona la app del banco. Sí, dice que no me carga. Sí, vale. Muchas gracias.

--- Audio 3 ---
Original: Texto original no disponible
Whisper:  Hola buena, la aplicación no se carga. La aplicación no carga el saldo de mi cuenta nueva. Dicen que la aplicación está siendo reparada y ahora no puedo acceder a mi cuenta.

--- Audio 4 ---
Original: Texto original no disponible
Whisper:  Hola, buenas. A ver, 

## 5. Prueba con Diferentes Modelos de Whisper

Whisper ofrece diferentes modelos con distintos niveles de precisión. Vamos a probar el modelo "whisper-1" (el estándar de la API) con uno de nuestros audios.

In [ ]:
from openai import OpenAI

# Cliente de OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))  # O usa `os.getenv("OPENAI_API_KEY")` si ya está configurada

def transcribe_audio(file_path, model="whisper-1"):
    with open(file_path, "rb") as f:
        response = client.audio.transcriptions.create(
            file=f,
            model=model
        )
    return response.text

In [ ]:
# Seleccionar un audio para probar
if len(transcriptions_df) > 0:
    test_audio = transcriptions_df.iloc[0]['file_path']
    original_text = transcriptions_df.iloc[0]['original_text']

    print(f"Audio seleccionado: {test_audio}")
    print(f"Texto original: {original_text}")

    # Transcribir con el modelo whisper-1
    try:
        whisper_result = transcribe_audio(test_audio, model="whisper-1")
        print(f"\nWhisper-1: {whisper_result}")

        if original_text != 'Texto original no disponible':
            similarity = calculate_similarity(original_text, whisper_result)
            print(f"Similitud: {similarity * 100:.2f}%")
    except Exception as e:
        print(f"Error al transcribir con whisper-1: {e}")

Audio seleccionado: audio_files/audio_5.wav
Texto original: Texto original no disponible

Whisper-1: Hola, buenas. Quiero utilizar la aplicación del banco, pero no consigo que funcione en condiciones. No sé si es que está en mantenimiento o tengo algún problema yo. ¿Me podríais ayudar?


## 6. Resumen de la Parte 2

En esta segunda parte del notebook, hemos:

1. Configurado las funciones necesarias para utilizar la API de Whisper
2. Transcrito los 5 audios aleatorios seleccionados en la Parte 1
3. Comparado las transcripciones con los textos originales
4. Analizado en detalle las diferencias entre textos originales y transcripciones
5. Probado el modelo estándar de Whisper en la API

En la Parte 3, utilizaremos un LLM para analizar las transcripciones y extraer información relevante.

# Caso 1: Audio to Text con Whisper y Datasets de Hugging Face (Parte 3A)

En esta tercera parte, utilizaremos un LLM para analizar las transcripciones obtenidas en la Parte 2 y extraer información relevante. Esta es la primera sección de la Parte 3, enfocada en la configuración y análisis de sentimiento.

## 1. Configuración para el Análisis con LLM

Primero, configuramos las bibliotecas y funciones necesarias para utilizar un LLM (GPT-3.5 o GPT-4) para analizar las transcripciones.

In [ ]:
# Función para llamar a la API de OpenAI
def get_completion(prompt, model="gpt-3.5-turbo", temperature=0, max_retries=3, retry_delay=5):
    for attempt in range(max_retries):
        try:
            messages = [{"role": "user", "content": prompt}]
            response = openai.ChatCompletion.create(
                model=model,
                messages=messages,
                temperature=temperature
            )
            return response.choices[0].message["content"]
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"Error: {e}. Reintentando en {retry_delay} segundos...")
                time.sleep(retry_delay)
            else:
                print(f"Error después de {max_retries} intentos: {e}")
                return f"Error: {e}"

## 2. Carga de Transcripciones

Vamos a cargar las transcripciones generadas en la Parte 2. Si estás ejecutando este notebook por separado, deberás crear un DataFrame con las transcripciones.

In [ ]:
# Intentar cargar transcriptions_df de la Parte 2
try:
    transcriptions_df
    print(f"DataFrame de transcripciones cargado con {len(transcriptions_df)} registros.")
except NameError:
    # Si no está definido, crear uno de ejemplo
    print("No se encontró el DataFrame de transcripciones. Creando uno de ejemplo...")

    # Ejemplo de transcripciones
    example_transcriptions = [
        {
            'index': 0,
            'file_path': 'audio_files/audio_1.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, buenas. Quiero utilizar la aplicación del banco, pero no consigo que funcione en condiciones. No sé si es que está en mantenimiento o tengo algún problema yo. ¿Me podríais ayudar?'
        },
        {
            'index': 1,
            'file_path': 'audio_files/audio_2.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, estoy intentando acceder a la aplicación de mi banco pero no carga correctamente. Me aparece un mensaje como que está siendo reparada y no puedo acceder.'
        },
        {
            'index': 2,
            'file_path': 'audio_files/audio_3.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, buenos días. Quería informaros porque no me funciona la app del banco. Sí, dice que no me carga. Sí, vale. Muchas gracias.'
        },
        {
            'index': 3,
            'file_path': 'audio_files/audio_4.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola buena, la aplicación no se carga. La aplicación no carga el saldo de mi cuenta nueva. Dicen que la aplicación está siendo reparada y ahora no puedo acceder a mi cuenta.'
        },
        {
            'index': 4,
            'file_path': 'audio_files/audio_5.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, buenas. A ver, tengo un problema con vuestra aplicación. Resulta que quiero hacer una transferencia bancaria a una cuenta conocida, pero me da error la aplicación. A ver qué puede ser.'
        }
    ]

    transcriptions_df = pd.DataFrame(example_transcriptions)
    print(f"DataFrame de ejemplo creado con {len(transcriptions_df)} registros.")

# Mostrar el DataFrame
transcriptions_df

DataFrame de transcripciones cargado con 5 registros.


## 3. Análisis de Sentimiento con LLM

Vamos a utilizar un LLM para analizar el sentimiento de cada transcripción.

In [ ]:
# Función para analizar el sentimiento con un LLM
def analyze_sentiment(text):
    prompt = f"""
    Analiza el sentimiento del siguiente texto en español y clasifícalo como positivo, negativo o neutral.
    Proporciona también una puntuación de confianza entre 0 y 1, y una breve explicación.
    Responde en formato JSON con las claves: sentiment, confidence, explanation.

    Texto: "{text}"
    """

    response = get_completion(prompt)

    try:
        # Intentar extraer el JSON de la respuesta
        # Primero, buscar si hay delimitadores de código
        if "```json" in response:
            json_str = response.split("```json")[1].split("```")[0].strip()
        elif "```" in response:
            json_str = response.split("```")[1].strip()
        else:
            json_str = response.strip()

        result = json.loads(json_str)
        return result
    except Exception as e:
        print(f"Error al parsear JSON: {e}")
        print(f"Respuesta original: {response}")
        return {
            "sentiment": "error",
            "confidence": 0,
            "explanation": f"Error: {e}"
        }

In [ ]:
from openai import OpenAI

# Cliente de OpenAI (puedes dejar la API key en una variable de entorno)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def analyze_sentiment(text):
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "Eres un analista de sentimientos. Clasifica el texto como positivo, negativo o neutral."},
                {"role": "user", "content": f"Texto: {text}"}
            ],
            temperature=0.2
        )

        content = response.choices[0].message.content

        # Intentamos parsear la respuesta (ajusta según cómo responde el modelo)
        if "positivo" in content.lower():
            sentiment = "positivo"
        elif "negativo" in content.lower():
            sentiment = "negativo"
        elif "neutral" in content.lower():
            sentiment = "neutral"
        else:
            sentiment = "desconocido"

        return {
            "sentiment": sentiment,
            "confidence": 1.0,  # Ajustable si parseas confianza explícitamente
            "explanation": content.strip()
        }
    except Exception as e:
        return {
            "sentiment": "error",
            "confidence": 0,
            "explanation": str(e)
        }

In [ ]:
# Analizar el sentimiento de cada transcripción
sentiment_results = []

for _, row in tqdm(transcriptions_df.iterrows(), total=len(transcriptions_df), desc="Analizando sentimiento"):
    text = row['transcribed_text']
    result = analyze_sentiment(text)

    sentiment_results.append({
        'index': row['index'],
        'text': text,
        'sentiment': result.get('sentiment', 'error'),
        'confidence': result.get('confidence', 0),
        'explanation': result.get('explanation', 'No disponible')
    })

# Crear DataFrame con los resultados
sentiment_df = pd.DataFrame(sentiment_results)
sentiment_df

Analizando sentimiento:   0%|          | 0/5 [00:00<?, ?it/s]

## 4. Visualización de Resultados de Sentimiento

Vamos a visualizar los resultados del análisis de sentimiento.

In [ ]:
# Visualizar la distribución de sentimientos
plt.figure(figsize=(10, 6))
sentiment_counts = sentiment_df['sentiment'].value_counts()

# Definir colores para cada sentimiento
colors = {'positivo': 'green', 'neutral': 'blue', 'negativo': 'red', 'error': 'gray'}
bar_colors = [colors.get(s, 'gray') for s in sentiment_counts.index]

# Crear gráfico de barras
bars = plt.bar(sentiment_counts.index, sentiment_counts.values, color=bar_colors)

# Añadir etiquetas
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             f'{height}', ha='center', va='bottom')

plt.xlabel('Sentimiento')
plt.ylabel('Cantidad')
plt.title('Distribución de Sentimientos en las Transcripciones')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar la confianza por sentimiento
plt.figure(figsize=(10, 6))

# Filtrar errores
valid_sentiments = sentiment_df[sentiment_df['sentiment'] != 'desconocido']

# Crear gráfico de barras agrupadas por sentimiento
sns.barplot(x='index', y='confidence', hue='sentiment', data=valid_sentiments, palette=colors)

plt.xlabel('Índice del Audio')
plt.ylabel('Confianza')
plt.title('Confianza del Análisis de Sentimiento por Transcripción')
plt.ylim(0, 1.05)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Resumen de la Parte 3A

En esta primera sección de la Parte 3, hemos:

1. Configurado las funciones necesarias para utilizar un LLM
2. Cargado las transcripciones generadas en la Parte 2
3. Analizado el sentimiento de cada transcripción con un LLM
4. Visualizado los resultados del análisis de sentimiento

En la Parte 3B, continuaremos con la extracción de entidades y la generación de resúmenes.

# Caso 1: Audio to Text con Whisper y Datasets de Hugging Face (Parte 3B)

En esta segunda sección de la Parte 3, continuaremos con el análisis de las transcripciones obtenidas en la Parte 2, enfocándonos en la extracción de entidades y generación de resúmenes.

In [ ]:
# Función para llamar a la API de OpenAI
def get_completion(prompt, model="gpt-3.5-turbo", temperature=0, max_retries=3, retry_delay=5):
    for attempt in range(max_retries):
        try:
            messages = [{"role": "user", "content": prompt}]
            response = openai.ChatCompletion.create(
                model=model,
                messages=messages,
                temperature=temperature
            )
            return response.choices[0].message["content"]
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"Error: {e}. Reintentando en {retry_delay} segundos...")
                time.sleep(retry_delay)
            else:
                print(f"Error después de {max_retries} intentos: {e}")
                return f"Error: {e}"

In [ ]:
# Cargar o crear transcriptions_df y sentiment_df si no están disponibles
try:
    transcriptions_df
    print(f"DataFrame de transcripciones ya cargado con {len(transcriptions_df)} registros.")
except NameError:
    print("Creando DataFrame de transcripciones de ejemplo...")
    example_transcriptions = [
        {
            'index': 0,
            'file_path': 'audio_files/audio_1.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, buenas. Quiero utilizar la aplicación del banco, pero no consigo que funcione en condiciones. No sé si es que está en mantenimiento o tengo algún problema yo. ¿Me podríais ayudar?'
        },
        {
            'index': 1,
            'file_path': 'audio_files/audio_2.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, estoy intentando acceder a la aplicación de mi banco pero no carga correctamente. Me aparece un mensaje como que está siendo reparada y no puedo acceder.'
        },
        {
            'index': 2,
            'file_path': 'audio_files/audio_3.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, buenos días. Quería informaros porque no me funciona la app del banco. Sí, dice que no me carga. Sí, vale. Muchas gracias.'
        },
        {
            'index': 3,
            'file_path': 'audio_files/audio_4.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola buena, la aplicación no se carga. La aplicación no carga el saldo de mi cuenta nueva. Dicen que la aplicación está siendo reparada y ahora no puedo acceder a mi cuenta.'
        },
        {
            'index': 4,
            'file_path': 'audio_files/audio_5.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, buenas. A ver, tengo un problema con vuestra aplicación. Resulta que quiero hacer una transferencia bancaria a una cuenta conocida, pero me da error la aplicación. A ver qué puede ser.'
        }
    ]
    transcriptions_df = pd.DataFrame(example_transcriptions)

try:
    sentiment_df
    print(f"DataFrame de sentimientos ya cargado con {len(sentiment_df)} registros.")
except NameError:
    print("Nota: sentiment_df no está disponible. Si necesitas los resultados del análisis de sentimiento, ejecuta primero la Parte 3.")

DataFrame de transcripciones ya cargado con 5 registros.
DataFrame de sentimientos ya cargado con 5 registros.


## 2. Extracción de Entidades con LLM

Ahora, utilizaremos un LLM para extraer entidades (personas, lugares, organizaciones, etc.) de cada transcripción.

In [ ]:
# Función para extraer entidades con un LLM
def extract_entities(text):
    prompt = f"""
    Extrae todas las entidades mencionadas en el siguiente texto en español.
    Clasifica cada entidad como: PERSONA, LUGAR, ORGANIZACIÓN, FECHA, HORA, u OTRO.
    Responde en formato JSON con una lista de objetos, cada uno con las claves: entity, type.
    Si no hay entidades, devuelve una lista vacía.

    Texto: "{text}"
    """

    response = get_completion(prompt)

    try:
        # Intentar extraer el JSON de la respuesta
        if "```json" in response:
            json_str = response.split("```json")[1].split("```")[0].strip()
        elif "```" in response:
            json_str = response.split("```")[1].strip()
        else:
            json_str = response.strip()

        result = json.loads(json_str)
        return result
    except Exception as e:
        print(f"Error al parsear JSON: {e}")
        print(f"Respuesta original: {response}")
        return []

In [ ]:
from openai import OpenAI

# Cliente de OpenAI (puedes usar una variable de entorno si ya configuraste la API key)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def extract_entities(text):
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "Eres un sistema de extracción de entidades. Extrae nombres de personas, lugares, organizaciones y fechas del texto. Devuelve una lista simple de entidades encontradas."},
                {"role": "user", "content": f"Texto: {text}"}
            ],
            temperature=0.2
        )

        content = response.choices[0].message.content.strip()

        # Intentamos extraer una lista desde el texto de salida (ajustable)
        entities = [line.strip("- ").strip() for line in content.splitlines() if line.strip()]
        return entities
    except Exception as e:
        return [f"Error: {str(e)}"]

In [ ]:
# Extraer entidades de cada transcripción
entity_results = []

for _, row in tqdm(transcriptions_df.iterrows(), total=len(transcriptions_df), desc="Extrayendo entidades"):
    text = row['transcribed_text']
    entities = extract_entities(text)

    entity_results.append({
        'index': row['index'],
        'text': text,
        'entities': entities,
        'entity_count': len(entities)
    })

# Crear DataFrame con los resultados
entity_df = pd.DataFrame(entity_results)
entity_df

Extrayendo entidades:   0%|          | 0/5 [00:00<?, ?it/s]

## 3. Visualización de Entidades Extraídas

Vamos a visualizar las entidades extraídas de las transcripciones.

In [ ]:
# Visualizar la cantidad de entidades por transcripción
plt.figure(figsize=(10, 6))
plt.bar(entity_df['index'], entity_df['entity_count'])
plt.xlabel('Índice del Audio')
plt.ylabel('Cantidad de Entidades')
plt.title('Cantidad de Entidades por Transcripción')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Extraer todas las entidades y sus tipos
all_entities = []
for _, row in entity_df.iterrows():
    for entity in row['entities']:
        all_entities.append({
            'entity': entity,
            'type': 'DESCONOCIDO'  # o una heurística si tienes info
        })

# Crear DataFrame con todas las entidades
if all_entities:
    all_entities_df = pd.DataFrame(all_entities)

    # Visualizar la distribución de tipos de entidades
    plt.figure(figsize=(10, 6))
    type_counts = all_entities_df['type'].value_counts()

    # Crear gráfico de barras
    bars = plt.bar(type_counts.index, type_counts.values)

    # Añadir etiquetas
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                 f'{height}', ha='center', va='bottom')

    plt.xlabel('Tipo de Entidad')
    plt.ylabel('Cantidad')
    plt.title('Distribución de Tipos de Entidades')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Mostrar las entidades más comunes
    print("Entidades más comunes:")
    print(all_entities_df['entity'].value_counts().head(10))
else:
    print("No se encontraron entidades en las transcripciones.")

Entidades más comunes:
entity
Entidades encontradas:                                           5
Lugares: No se encontraron lugares en el texto.                  3
Fechas: No se encontraron fechas en el texto.                    3
Organizaciones: No se encontraron organizaciones en el texto.    2
Personas: No se encontraron personas en el texto.                2
Organizaciones: Banco                                            1
Banco (organización)                                             1
Ninguna                                                          1
Personas: No se encontraron nombres de personas en el texto.     1
Name: count, dtype: int64


## 4. Resumen de la Parte 3B

En esta segunda sección de la Parte 3, hemos:

1. Configurado el entorno y cargado los datos necesarios
2. Extraído entidades de cada transcripción con un LLM
3. Visualizado las entidades extraídas y su distribución por tipos

En la Parte 3C, continuaremos con la generación de resúmenes y la integración de todos los resultados.

In [ ]:
# Cargar o crear DataFrames si no están disponibles
try:
    transcriptions_df
    print(f"DataFrame de transcripciones ya cargado con {len(transcriptions_df)} registros.")
except NameError:
    print("Creando DataFrame de transcripciones de ejemplo...")
    example_transcriptions = [
        {
            'index': 0,
            'file_path': 'audio_files/audio_1.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, buenas. Quiero utilizar la aplicación del banco, pero no consigo que funcione en condiciones. No sé si es que está en mantenimiento o tengo algún problema yo. ¿Me podríais ayudar?'
        },
        {
            'index': 1,
            'file_path': 'audio_files/audio_2.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, estoy intentando acceder a la aplicación de mi banco pero no carga correctamente. Me aparece un mensaje como que está siendo reparada y no puedo acceder.'
        },
        {
            'index': 2,
            'file_path': 'audio_files/audio_3.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, buenos días. Quería informaros porque no me funciona la app del banco. Sí, dice que no me carga. Sí, vale. Muchas gracias.'
        },
        {
            'index': 3,
            'file_path': 'audio_files/audio_4.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola buena, la aplicación no se carga. La aplicación no carga el saldo de mi cuenta nueva. Dicen que la aplicación está siendo reparada y ahora no puedo acceder a mi cuenta.'
        },
        {
            'index': 4,
            'file_path': 'audio_files/audio_5.wav',
            'original_text': 'Texto original no disponible',
            'transcribed_text': 'Hola, buenas. A ver, tengo un problema con vuestra aplicación. Resulta que quiero hacer una transferencia bancaria a una cuenta conocida, pero me da error la aplicación. A ver qué puede ser.'
        }
    ]
    transcriptions_df = pd.DataFrame(example_transcriptions)

try:
    sentiment_df
    print(f"DataFrame de sentimientos ya cargado con {len(sentiment_df)} registros.")
except NameError:
    print("Nota: sentiment_df no está disponible. Si necesitas los resultados del análisis de sentimiento, ejecuta primero la Parte 3.")

try:
    entity_df
    print(f"DataFrame de entidades ya cargado con {len(entity_df)} registros.")
except NameError:
    print("Nota: entity_df no está disponible. Si necesitas los resultados de la extracción de entidades, ejecuta primero la Parte 3.")

DataFrame de transcripciones ya cargado con 5 registros.
DataFrame de sentimientos ya cargado con 5 registros.
DataFrame de entidades ya cargado con 5 registros.


## 2. Generación de Resúmenes con LLM

Utilizaremos un LLM para generar un resumen de cada transcripción.

In [ ]:
# Función para generar un resumen con un LLM
def generate_summary(text):
    prompt = f"""
    Genera un resumen conciso del siguiente texto en español.
    El resumen debe capturar los puntos principales y no debe exceder las 3 oraciones.

    Texto: "{text}"
    """

    response = get_completion(prompt)
    return response.strip()

In [ ]:
from openai import OpenAI

# Cliente
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def generate_summary(text):
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {
                    "role": "system",
                    "content": "Eres un asistente que resume textos de forma clara y concisa."
                },
                {
                    "role": "user",
                    "content": f"Resume el siguiente texto en pocas frases:\n\n{text}"
                }
            ],
            temperature=0.3,
            max_tokens=300
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error al generar resumen: {str(e)}"

In [ ]:
# Generar resúmenes de cada transcripción
summary_results = []

for _, row in tqdm(transcriptions_df.iterrows(), total=len(transcriptions_df), desc="Generando resúmenes"):
    text = row['transcribed_text']
    summary = generate_summary(text)

    summary_results.append({
        'index': row['index'],
        'text': text,
        'summary': summary,
        'summary_length': len(summary.split())
    })

# Crear DataFrame con los resultados
summary_df = pd.DataFrame(summary_results)
summary_df

Generando resúmenes:   0%|          | 0/5 [00:00<?, ?it/s]

## 3. Visualización de Resultados de Resúmenes

Vamos a visualizar la longitud de los resúmenes generados.

In [ ]:
# Visualizar la longitud de los resúmenes
plt.figure(figsize=(10, 6))
plt.bar(summary_df['index'], summary_df['summary_length'])
plt.xlabel('Índice del Audio')
plt.ylabel('Longitud del Resumen (palabras)')
plt.title('Longitud de los Resúmenes Generados')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Integración de Resultados

Vamos a integrar todos los resultados en un único DataFrame para tener una visión completa del análisis.

In [ ]:
# Integrar todos los resultados
integrated_results = []

for i in range(len(transcriptions_df)):
    # Obtener datos de cada DataFrame
    transcription = transcriptions_df[transcriptions_df['index'] == i].iloc[0] if i in transcriptions_df['index'].values else None

    # Intentar obtener datos de sentiment_df si está disponible
    try:
        sentiment = sentiment_df[sentiment_df['index'] == i].iloc[0] if i in sentiment_df['index'].values else None
    except NameError:
        sentiment = None

    # Intentar obtener datos de entity_df si está disponible
    try:
        entity = entity_df[entity_df['index'] == i].iloc[0] if i in entity_df['index'].values else None
    except NameError:
        entity = None

    summary = summary_df[summary_df['index'] == i].iloc[0] if i in summary_df['index'].values else None

    if transcription is not None:
        result = {
            'index': i,
            'original_text': transcription['original_text'],
            'transcribed_text': transcription['transcribed_text'],
            'sentiment': sentiment['sentiment'] if sentiment is not None else None,
            'confidence': sentiment['confidence'] if sentiment is not None else None,
            'entity_count': entity['entity_count'] if entity is not None else 0,
            'summary': summary['summary'] if summary is not None else None
        }
        integrated_results.append(result)

# Crear DataFrame integrado
integrated_df = pd.DataFrame(integrated_results)
integrated_df

## 5. Exportar los Resultados

Vamos a exportar los resultados integrados a un archivo CSV para su uso posterior.

In [ ]:
# Exportar resultados a CSV
output_dir = "resultados"
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, "resultados_analisis.csv")

integrated_df.to_csv(output_file, index=False)
print(f"Resultados exportados a: {output_file}")

Resultados exportados a: resultados/resultados_analisis.csv


## 7. Conclusiones Generales

A lo largo de las tres partes de este notebook, hemos explorado el proceso completo de Audio to Text con Whisper y análisis con LLMs:

1. **Parte 1**: Carga y preparación de datasets de Hugging Face, seleccionando 5 audios aleatorios.
2. **Parte 2**: Transcripción de audio con Whisper y evaluación de la precisión.
3. **Parte 3**: Análisis de las transcripciones con LLMs para extraer sentimiento, entidades y generar resúmenes.

Este flujo de trabajo demuestra cómo se pueden combinar diferentes tecnologías de IA para procesar y analizar audio de manera efectiva, sin necesidad de filtrar por duración y manteniendo el código modular para evitar problemas de truncamiento en Colab.

Las aplicaciones prácticas de este flujo son numerosas, desde la transcripción de reuniones y entrevistas hasta el análisis de llamadas de servicio al cliente, la creación de subtítulos automáticos, o cualquier otra aplicación que requiera convertir audio a texto y extraer información valiosa del contenido.